# Multi-Seed Validation — Gemma-2-9B | CB1 | Stage 4
## Seeds: 123 and 456 (seed=42 already complete: F1=0.9082)
**Purpose:** Validate that the champion result (F1=0.9082) is robust across seeds, not random variance.

**Protocol:**
- Data split: **always seed=42** (same test set as all stages)
- Training seed: **123, 456** (changes weight init, dropout, data shuffling)
- Model: `google/gemma-2-9b-it` | Rank: r=4 | Formula: Wf = W(I+UV)+AB
- After both runs: report mean ± std across all 3 seeds (42, 123, 456)

**Expected result in paper:**
```
Seed 42:  F1 = 0.9082 (already completed)
Seed 123: F1 = 0.90xx
Seed 456: F1 = 0.90xx
Mean ± std: 0.90xx ± 0.00xx → consistently above Stage 2 (0.9010)
```


---
## Step 1: Install

**After running: Runtime → Restart Runtime, then Step 2 onwards.**

In [1]:
!pip install -q \
    "transformers>=4.47.0,<4.52.0" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.0.0" \
    "numpy<2.1" \
    "pandas<3.0" \
    "torchvision==0.20.1" \
    "scikit-learn" \
    tqdm matplotlib seaborn

print("✓ Installation complete. Restart runtime, then run from Step 2.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## Step 2: Imports

In [1]:
import os, random, collections, re
from typing import Dict, List
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    balanced_accuracy_score, matthews_corrcoef,
)
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments, Trainer, TrainerCallback,
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
import bitsandbytes.functional as bnb_F
import bitsandbytes as bnb
from google.colab import drive, userdata

def set_all_seeds(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(42)
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print("✓ HF token loaded.")
drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device} | {torch.cuda.get_device_name(0)}")


✓ HF token loaded.
Mounted at /content/drive
✓ Device: cuda | NVIDIA A100-SXM4-80GB


---
## Step 3: Configuration

In [2]:
MODEL_NAME   = "google/gemma-2-9b-it"
NUM_LABELS   = 6
MAX_LENGTH   = 256
LORA_RANK    = 4       # Champion rank from Stage 4 CB1
LORA_ALPHA   = 4
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

NUM_EPOCHS       = 1
LEARNING_RATE    = 2e-4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE  = 16
WEIGHT_DECAY     = 0.01
EVAL_STEPS       = 200
LOGGING_STEPS    = 50
OUTPUT_DIR       = "/content/drive/MyDrive/Results/CB1_GeneralisedLoRA_Gemma2_9B_MultiSeed"

CB1_PATH             = "/content/drive/MyDrive/Datasets/cb1.csv"
SPLIT_SEED           = 42    # ALWAYS 42 — same test set as all stages
SIMILARITY_THRESHOLD = 0.90
TRAIN_SEEDS          = [123, 456]  # Seed 42 already done (F1=0.9082)

LABEL_MAPPING = {
    'not_cyberbullying': 0, 'gender': 1, 'religion': 2,
    'ethnicity': 3, 'age': 4, 'other_cyberbullying': 5,
}
ID2LABEL     = {v: k for k, v in LABEL_MAPPING.items()}
TARGET_NAMES = list(LABEL_MAPPING.keys())

print("✓ Multi-seed validation config loaded.")
print(f"  Model      : {MODEL_NAME}")
print(f"  Rank       : r={LORA_RANK} (champion)")
print(f"  Split seed : {SPLIT_SEED} (fixed — same test set)")
print(f"  Train seeds: {TRAIN_SEEDS} (seed=42 already done)")
print(f"  Seed 42 result: F1=0.9082 | S2 target: 0.9010")


✓ Multi-seed validation config loaded.
  Model      : google/gemma-2-9b-it
  Rank       : r=4 (champion)
  Split seed : 42 (fixed — same test set)
  Train seeds: [123, 456] (seed=42 already done)
  Seed 42 result: F1=0.9082 | S2 target: 0.9010


---
## Step 4–6: Preprocessing, Loading, Splitting

Identical to all stages — split uses seed=42 (fixed).

In [3]:
def preprocess_text_for_llm(text, remove_hashtags=False, remove_emojis=False):
    if not isinstance(text, str): return ""
    text = re.sub(r'\bRT\s+@\w+:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    if remove_hashtags: text = re.sub(r'#(\w+)', r'\1', text)
    if remove_emojis: text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'([!?.])\1+', r'\1', text)
    if len(text.split()) < 3: return ""
    return text

# Load and clean
df = pd.read_csv(CB1_PATH)
df = df.rename(columns={'tweet_text': 'text', 'cyberbullying_type': 'label'})
corrupt_mask = df['text'].astype(str).str.strip().isin(['#NAME?','#REF!','#VALUE!','#N/A','nan',''])
df = df[~corrupt_mask].reset_index(drop=True)
df['text'] = df['text'].apply(preprocess_text_for_llm)
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)
text_label_counts = df.groupby('text')['label'].nunique()
conflicting = text_label_counts[text_label_counts > 1].index
df = df[~df['text'].isin(conflicting)].reset_index(drop=True)
df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
df = df[df['text'].str.split().str.len() >= 3].reset_index(drop=True)
print(f"Dataset: {len(df):,} samples")

# Split with FIXED seed=42 — same test set as all stages
train_val_df, test_df = train_test_split(
    df, test_size=0.25, random_state=SPLIT_SEED, stratify=df['label'])

all_texts = list(train_val_df['text'].values) + list(test_df['text'].values)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)
tfidf_matrix = vectorizer.fit_transform(all_texts)
train_vecs = tfidf_matrix[:len(train_val_df)]
test_vecs = tfidf_matrix[len(train_val_df):]
leaked_indices = []
for start in range(0, len(test_df), 500):
    end = min(start+500, len(test_df))
    batch_sims = cosine_similarity(test_vecs[start:end], train_vecs)
    leaked_mask = batch_sims.max(axis=1) >= SIMILARITY_THRESHOLD
    leaked_indices.extend(test_df.index[start:end][leaked_mask].tolist())
if leaked_indices:
    leaked = test_df.loc[leaked_indices]
    test_df = test_df.drop(leaked_indices).reset_index(drop=True)
    train_val_df = pd.concat([train_val_df, leaked], ignore_index=True)
else:
    train_val_df = train_val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.10, random_state=SPLIT_SEED, stratify=train_val_df['label'])
train_df = train_df.reset_index(drop=True); val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"✓ Split seed={SPLIT_SEED} — identical test set to all stages.")


Dataset: 43,334 samples
Train: 29,703 | Val: 3,301 | Test: 10,330
✓ Split seed=42 — identical test set to all stages.


---
## Step 7–10: Helpers, Metrics, Stability, Trainer

In [4]:
def get_class_weights(df, label_col='label'):
    counts = df[label_col].value_counts()
    total = len(df); n_classes = len(LABEL_MAPPING)
    weights = [total / (n_classes * counts.get(ID2LABEL[i], 1)) for i in range(n_classes)]
    return [w / sum(weights) * n_classes for w in weights]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average='macro', zero_division=0),
        "mcc": matthews_corrcoef(labels, predictions),
    }

class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32) if class_weights is not None else None
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels"); outputs = model(**inputs); logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device)) if self.class_weights is not None else nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✓ Helpers defined.")


✓ Helpers defined.


---
## Step 11: GeneralisedLoRALayer + Training Function

In [5]:
class GeneralisedLoRALayer(nn.Module):
    def __init__(self, base_layer, rank, dropout=0.05, alpha=None):
        super().__init__()
        self.rank = rank
        self.alpha = alpha if alpha is not None else rank
        self.scaling = self.alpha / self.rank
        inner = base_layer.base_layer if hasattr(base_layer, 'base_layer') else base_layer
        if hasattr(inner, 'in_features'):
            n, m = inner.in_features, inner.out_features
        elif hasattr(inner, 'weight'):
            n, m = inner.weight.shape[1], inner.weight.shape[0]
        else:
            raise ValueError(f"Cannot determine dims from {type(inner)}")
        self.n = n; self.m = m; self.inner = inner
        _dev = self.inner.weight.device
        self.lora_U = nn.Parameter(torch.empty(n, rank, device=_dev))
        self.lora_V = nn.Parameter(torch.zeros(rank, n, device=_dev))
        self.lora_A = nn.Parameter(torch.empty(m, rank, device=_dev))
        self.lora_B = nn.Parameter(torch.zeros(rank, n, device=_dev))
        nn.init.normal_(self.lora_U, mean=0.0, std=0.01)
        nn.init.normal_(self.lora_A, mean=0.0, std=0.01)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()
        for param in self.inner.parameters(): param.requires_grad = False

    def forward(self, x):
        base_out = self.inner(x)
        x_drop = self.dropout(x)
        lora_vx = x_drop @ self.lora_V.T
        lora_uvx = lora_vx @ self.lora_U.T
        mult_out = self.inner(lora_uvx)
        lora_bx = x_drop @ self.lora_B.T
        add_out = lora_bx @ self.lora_A.T
        return base_out + self.scaling * mult_out + self.scaling * add_out


def apply_generalised_lora_over_peft(model, target_modules, rank, dropout=0.05, alpha=None):
    from peft.tuners.lora import LoraLayer
    if alpha is None: alpha = rank
    replaced = 0
    module_dict = dict(model.named_modules())
    for name, module in list(model.named_modules()):
        if not isinstance(module, LoraLayer): continue
        if not any(name.endswith(f'.{t}') or name == t for t in target_modules): continue
        parts = name.rsplit('.', 1)
        if len(parts) == 2:
            parent = module_dict.get(parts[0])
            if parent is None: continue
            attr_name = parts[1]
        else:
            parent = model; attr_name = name
        setattr(parent, attr_name, GeneralisedLoRALayer(module, rank=rank, dropout=dropout, alpha=alpha))
        replaced += 1
    print(f"  ✓ Replaced {replaced} layers")
    return model


def run_single_seed(train_df, val_df, test_df, train_seed):
    """Run one training with a specific seed. Split is fixed at seed=42."""
    set_all_seeds(train_seed)
    label = f'seed_{train_seed}'
    print(f'\n{"="*65}')
    print(f'  MULTI-SEED RUN — train_seed={train_seed}')
    print(f'  Split seed=42 (fixed) | Model: Gemma-2-9B | r={LORA_RANK}')
    print(f'{"="*65}')

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_dir = os.path.join(OUTPUT_DIR, label)
    os.makedirs(run_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    def tokenise(examples):
        return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

    train_m = train_df.copy(); val_m = val_df.copy(); test_m = test_df.copy()
    for s in [train_m, val_m, test_m]:
        s['label'] = s['label'].map(LABEL_MAPPING)
        assert s['label'].notnull().all()

    hf_train = Dataset.from_pandas(train_m[['text','label']].reset_index(drop=True)).map(tokenise, batched=True, remove_columns=['text'])
    hf_val = Dataset.from_pandas(val_m[['text','label']].reset_index(drop=True)).map(tokenise, batched=True, remove_columns=['text'])
    hf_train.set_format('torch'); hf_val.set_format('torch')

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT, bias='none', task_type='SEQ_CLS')

    print('[1] Loading model...')
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, quantization_config=quant_config, num_labels=NUM_LABELS,
        torch_dtype=torch.bfloat16, device_map='auto', token=HF_TOKEN)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)

    print('[2] Injecting GeneralisedLoRALayer...')
    model = apply_generalised_lora_over_peft(
        model, TARGET_MODULES, rank=LORA_RANK, dropout=LORA_DROPOUT, alpha=LORA_ALPHA)

    class_weights = get_class_weights(train_df)

    training_args = TrainingArguments(
        output_dir=run_dir, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        eval_strategy='steps', eval_steps=EVAL_STEPS,
        logging_steps=LOGGING_STEPS, save_strategy='no',
        fp16=False, bf16=True, report_to='none', remove_unused_columns=False)

    trainer = CustomTrainer(
        model=model, args=training_args,
        train_dataset=hf_train, eval_dataset=hf_val,
        compute_metrics=compute_metrics, class_weights=class_weights)

    print(f'[3] Training (seed={train_seed})...')
    train_result = trainer.train()
    runtime = train_result.metrics.get('train_runtime', 0)
    print(f'    Time: {runtime:.0f}s ({runtime/3600:.2f} hrs)')

    # Evaluate
    print('[4] Evaluating...')
    model.eval()
    dev = next(model.parameters()).device
    all_preds = []
    texts = test_m['text'].tolist()
    for start in range(0, len(texts), 32):
        enc = tokenizer(texts[start:start+32], padding='max_length',
                        truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(dev)
        with torch.no_grad():
            logits = model(**enc).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy().tolist())

    y_true = test_m['label'].tolist()
    macro_f1 = f1_score(y_true, all_preds, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, all_preds)
    accuracy = accuracy_score(y_true, all_preds)

    result = {'seed': train_seed, 'macro_f1': macro_f1, 'mcc': mcc,
              'accuracy': accuracy, 'train_runtime_s': runtime}

    pd.DataFrame([result]).to_csv(os.path.join(run_dir, f'result_seed{train_seed}.csv'), index=False)

    print(f'\n    Seed {train_seed}: F1={macro_f1:.4f} | MCC={mcc:.4f} | Acc={accuracy:.4f}')

    del model, trainer; torch.cuda.empty_cache()
    return result


print("✓ run_single_seed() defined.")


✓ run_single_seed() defined.


---
## Step 12: Run Seeds 123 and 456

Seed 42 already done (F1=0.9082). Each run ~3.5 hrs.


In [6]:
all_results = []

# Seed 42 — already completed, hardcode the result
all_results.append({
    'seed': 42, 'macro_f1': 0.9082, 'mcc': 0.9019, 'accuracy': 0.9198,
    'train_runtime_s': 0, 'note': 'from original Stage 4 CB1 run'
})
print("Seed 42: F1=0.9082 (from original run — not re-trained)")

# Run seeds 123 and 456
for seed in TRAIN_SEEDS:
    result = run_single_seed(train_df, val_df, test_df, train_seed=seed)
    all_results.append(result)


Seed 42: F1=0.9082 (from original run — not re-trained)

  MULTI-SEED RUN — train_seed=123
  Split seed=42 (fixed) | Model: Gemma-2-9B | r=4


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

[2] Injecting GeneralisedLoRALayer...
  ✓ Replaced 168 layers
[3] Training (seed=456)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,F1,Mcc
200,0.500200,1.026874,0.818237,0.784737,0.784006
400,0.686000,0.641178,0.841866,0.820439,0.816116
600,0.687400,0.871304,0.832778,0.780906,0.807333
800,0.821600,0.643975,0.837928,0.816135,0.811101
1000,0.413700,0.638064,0.871554,0.852753,0.848027
1200,0.730000,0.569911,0.868525,0.847310,0.846170
1400,0.678900,0.631770,0.858831,0.837958,0.835390
1600,0.652900,0.590084,0.880036,0.860007,0.858137
1800,0.486500,0.521782,0.877916,0.862654,0.853612
2000,0.404800,0.601384,0.879733,0.866123,0.858991


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

    Time: 12374s (3.44 hrs)
[4] Evaluating...

    Seed 456: F1=0.9039 | MCC=0.8976 | Acc=0.9148


---
## Step 13: Multi-Seed Summary

In [7]:
print('\n' + '═'*70)
print('  MULTI-SEED VALIDATION — Gemma-2-9B CB1 Stage 4')
print('  Formula: Wf = W(I + UV) + AB | r=4')
print('  Split seed: 42 (fixed) | Train seeds: 42, 123, 456')
print('═'*70)

f1_values = [r['macro_f1'] for r in all_results]
mcc_values = [r['mcc'] for r in all_results]

print(f'\n  {"Seed":<8} {"Macro F1":<12} {"MCC":<12}')
print(f'  {"─"*32}')
for r in all_results:
    note = ' (original)' if r['seed'] == 42 else ''
    print(f'  {r["seed"]:<8} {r["macro_f1"]:<12.4f} {r["mcc"]:<12.4f}{note}')

mean_f1 = np.mean(f1_values)
std_f1  = np.std(f1_values)
mean_mcc = np.mean(mcc_values)
std_mcc  = np.std(mcc_values)

print(f'\n  {"─"*32}')
print(f'  Mean     {mean_f1:<12.4f} {mean_mcc:<12.4f}')
print(f'  Std      {std_f1:<12.4f} {std_mcc:<12.4f}')
print(f'  Range    {max(f1_values)-min(f1_values):<12.4f} {max(mcc_values)-min(mcc_values):<12.4f}')

print(f'\n  ── Comparison ──')
print(f'  Stage 2 best (seed 42): F1=0.9010')
print(f'  Stage 3 best (seed 42): F1=0.8944')
print(f'  Stage 4 mean ± std    : F1={mean_f1:.4f} ± {std_f1:.4f}')
print(f'  Stage 4 min           : F1={min(f1_values):.4f}')

if min(f1_values) > 0.9010:
    print(f'\n  ★ ALL seeds beat Stage 2 (0.9010) — robust improvement confirmed!')
elif mean_f1 > 0.9010:
    print(f'\n  ✓ Mean beats Stage 2 (0.9010) — improvement consistent on average.')
else:
    print(f'\n  → Mean below Stage 2 — improvement may not be robust.')

if min(f1_values) > 0.8944:
    print(f'  ★ ALL seeds beat Stage 3 (0.8944)')

# Save combined CSV
summary_df = pd.DataFrame(all_results)
summary_df.loc[len(summary_df)] = {
    'seed': 'mean', 'macro_f1': mean_f1, 'mcc': mean_mcc,
    'accuracy': np.mean([r['accuracy'] for r in all_results]),
    'train_runtime_s': 0,
}
summary_df.loc[len(summary_df)] = {
    'seed': 'std', 'macro_f1': std_f1, 'mcc': std_mcc,
    'accuracy': np.std([r['accuracy'] for r in all_results]),
    'train_runtime_s': 0,
}
summary_path = os.path.join(OUTPUT_DIR, 'multi_seed_summary_CB1_FINAL.csv')
summary_df.to_csv(summary_path, index=False)
print(f'\n✓ Results saved → {summary_path}')



══════════════════════════════════════════════════════════════════════
  MULTI-SEED VALIDATION — Gemma-2-9B CB1 Stage 4
  Formula: Wf = W(I + UV) + AB | r=4
  Split seed: 42 (fixed) | Train seeds: 42, 123, 456
══════════════════════════════════════════════════════════════════════

  Seed     Macro F1     MCC         
  ────────────────────────────────
  42       0.9082       0.9019       (original)
  123      0.9054       0.8990      
  456      0.9039       0.8976      

  ────────────────────────────────
  Mean     0.9058       0.8995      
  Std      0.0018       0.0018      
  Range    0.0043       0.0043      

  ── Comparison ──
  Stage 2 best (seed 42): F1=0.9010
  Stage 3 best (seed 42): F1=0.8944
  Stage 4 mean ± std    : F1=0.9058 ± 0.0018
  Stage 4 min           : F1=0.9039

  ★ ALL seeds beat Stage 2 (0.9010) — robust improvement confirmed!
  ★ ALL seeds beat Stage 3 (0.8944)

✓ Results saved → /content/drive/MyDrive/Results/CB1_GeneralisedLoRA_Gemma2_9B_MultiSeed/multi_se